# Predict Podcast Listening Time
Kaggle Competition Link: https://www.kaggle.com/competitions/playground-series-s5e4/overview

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import layers, models

## Data Loading

In [2]:
# Loading Training and Testing Datasets and Sample Submission File from links
df =  pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/train.csv")
df_test = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/test.csv")
df_sample = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/sample_submission.csv")
df.head()

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   Episode_Title                750000 non-null  object 
 3   Episode_Length_minutes       662907 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  603970 non-null  float64
 9   Number_of_Ads                749999 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 68.7+ MB


In [4]:
df.describe()

,id,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Listening_Time_minutes
count,750000.000000,662907.000000,750000.000000,603970.000000,749999.000000,750000.000000
mean,374999.500000,64.504738,59.859901,52.236449,1.348855,45.437406
std,216506.495284,32.969603,22.873098,28.451241,1.151130,27.138306
min,0.000000,0.000000,1.300000,0.000000,0.000000,0.000000
25%,187499.750000,35.730000,39.410000,28.380000,0.000000,23.178350
50%,374999.500000,63.840000,60.050000,53.580000,1.000000,43.379460
75%,562499.250000,94.070000,79.530000,76.600000,2.000000,64.811580
max,749999.000000,325.240000,119.460000,119.910000,103.910000,119.970000


In [5]:
# Check unique element for each column
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

id: [     0      1      2 ... 749997 749998 749999]
Podcast_Name: ['Mystery Matters' 'Joke Junction' 'Study Sessions' 'Digital Digest'
 'Mind & Body' 'Fitness First' 'Criminal Minds' 'News Roundup'
 'Daily Digest' 'Music Matters' 'Sports Central' 'Melody Mix' 'Game Day'
 'Gadget Geek' 'Global News' 'Tech Talks' 'Sport Spot' 'Funny Folks'
 'Sports Weekly' 'Business Briefs' 'Tech Trends' 'Innovators'
 'Health Hour' 'Comedy Corner' 'Sound Waves' 'Brain Boost'
 "Athlete's Arena" 'Wellness Wave' 'Style Guide' 'World Watch' 'Humor Hub'
 'Money Matters' 'Healthy Living' 'Home & Living' 'Educational Nuggets'
 'Market Masters' 'Learning Lab' 'Lifestyle Lounge' 'Crime Chronicles'
 'Detective Diaries' 'Life Lessons' 'Current Affairs' 'Finance Focus'
 'Laugh Line' 'True Crime Stories' 'Business Insights' 'Fashion Forward'
 'Tune Time']
Episode_Title: ['Episode 98' 'Episode 26' 'Episode 16' 'Episode 45' 'Episode 86'
 'Episode 19' 'Episode 47' 'Episode 44' 'Episode 32' 'Episode 81'
 'Episode 66' 'Ep

In [6]:
df.isna().sum()

,0
id,0
Podcast_Name,0
Episode_Title,0
Episode_Length_minutes,87093
Genre,0
Host_Popularity_percentage,0
Publication_Day,0
Publication_Time,0
Guest_Popularity_percentage,146030
Number_of_Ads,1


In [7]:
df.duplicated().sum()

np.int64(0)

## Data Cleaning

In [8]:
# Check rows where Episode_Length_minutes > 300
df_test[df_test["Episode_Length_minutes"] > 300]

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
54434,804434,Current Affairs,Episode 36,7575.0,News,89.54,Saturday,Night,NaN,2.0,Negative
56597,806597,Market Masters,Episode 23,78486264.0,Business,55.45,Monday,Evening,48.5,0.0,Positive


In [9]:
# Change the value for rows with Episode_Length_minutes with the mean of Episode_Length_minutes based on the Podcast Name excluding such rows
df_test.loc[df_test["Episode_Length_minutes"] > 300, "Episode_Length_minutes"] = df_test.loc[df_test["Episode_Length_minutes"] > 300, "Podcast_Name"].map(df.groupby("Podcast_Name")["Episode_Length_minutes"].mean())

In [10]:
df_test.iloc[[54434, 56597]]

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
54434,804434,Current Affairs,Episode 36,62.010221,News,89.54,Saturday,Night,NaN,2.0,Negative
56597,806597,Market Masters,Episode 23,65.207962,Business,55.45,Monday,Evening,48.5,0.0,Positive


In [11]:
# Create column that distinguishes train data with '1' or '0' for test. Combine datasets after
df["is_train"] = 1
df_test["is_train"] = 0
df_combined = pd.concat([df, df_test])

In [12]:
df_combined.head()

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,is_train
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998,1
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241,1
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531,1
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824,1
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031,1


In [13]:
df_combined.describe()

,id,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Listening_Time_minutes,is_train
count,1000000.000000,884171.000000,1000000.000000,805138.000000,999999.000000,750000.000000,1000000.000000
mean,499999.500000,64.515402,59.824048,52.225542,1.350604,45.437406,0.750000
std,288675.278933,32.965418,22.874903,28.449679,2.358272,27.138306,0.433013
min,0.000000,0.000000,1.300000,0.000000,0.000000,0.000000,0.000000
25%,249999.750000,35.740000,39.370000,28.370000,0.000000,23.178350,0.750000
50%,499999.500000,63.870000,60.020000,53.540000,1.000000,43.379460,1.000000
75%,749999.250000,94.080000,79.490000,76.590000,2.000000,64.811580,1.000000
max,999999.000000,325.240000,119.460000,119.910000,2063.000000,119.970000,1.000000


In [14]:
# Fill null values for Episode_Length_minutes, Guest_Popularity_percentage, and Number_of_Ads based on Podcast Name
df_combined["Episode_Length_minutes"] = df_combined.groupby("Podcast_Name")["Episode_Length_minutes"].transform(lambda x: x.fillna(x.mean()))
df_combined["Guest_Popularity_percentage"] = df_combined.groupby("Podcast_Name")["Guest_Popularity_percentage"].transform(lambda x: x.fillna(x.mean()))
df_combined["Number_of_Ads"] = df_combined.groupby("Podcast_Name")["Number_of_Ads"].transform(lambda x: x.fillna(x.mean()))

In [15]:
df_combined.describe()

,id,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Listening_Time_minutes,is_train
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,750000.000000,1000000.000000
mean,499999.500000,64.512580,59.824048,52.222469,1.350604,45.437406,0.750000
std,288675.278933,31.002671,22.874903,25.530172,2.358271,27.138306,0.433013
min,0.000000,0.000000,1.300000,0.000000,0.000000,0.000000,0.000000
25%,249999.750000,39.400000,39.370000,34.550000,0.000000,23.178350,0.750000
50%,499999.500000,64.400141,60.020000,52.429664,1.000000,43.379460,1.000000
75%,749999.250000,90.340000,79.490000,71.030000,2.000000,64.811580,1.000000
max,999999.000000,325.240000,119.460000,119.910000,2063.000000,119.970000,1.000000


In [16]:
df_combined.duplicated().sum()

np.int64(0)

In [17]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   id                           1000000 non-null  int64  
 1   Podcast_Name                 1000000 non-null  object 
 2   Episode_Title                1000000 non-null  object 
 3   Episode_Length_minutes       1000000 non-null  float64
 4   Genre                        1000000 non-null  object 
 5   Host_Popularity_percentage   1000000 non-null  float64
 6   Publication_Day              1000000 non-null  object 
 7   Publication_Time             1000000 non-null  object 
 8   Guest_Popularity_percentage  1000000 non-null  float64
 9   Number_of_Ads                1000000 non-null  float64
 10  Episode_Sentiment            1000000 non-null  object 
 11  Listening_Time_minutes       750000 non-null   float64
 12  is_train                     1000000 non-null  i

In [18]:
numerical_cols = df_combined.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_combined.select_dtypes(include=[object]).columns.tolist()
# Remove is_train, id
numerical_cols.remove("is_train")
numerical_cols.remove("id")
numerical_cols.remove("Listening_Time_minutes")

In [19]:
numerical_cols

['Episode_Length_minutes',
 'Host_Popularity_percentage',
 'Guest_Popularity_percentage',
 'Number_of_Ads']

## Data Preprocessing

In [20]:
def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

In [21]:
def preprocess_combined_df(df_combined):
    df = df_combined.copy()

    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=[object]).columns.tolist()

    for col in ['is_train', 'id', 'Listening_Time_minutes']:
        if col in numerical_cols:
            numerical_cols.remove(col)

    encoders = {col: LabelEncoder().fit(df[col]) for col in categorical_cols}
    for col in categorical_cols:
        df[col + '_id'] = encoders[col].transform(df[col])

    scalers = {col: StandardScaler().fit(df[df['is_train'] == 1][[col]]) for col in numerical_cols}
    for col in numerical_cols:
        df[col + '_scaled'] = scalers[col].transform(df[[col]])

    cat_ids = [col + '_id' for col in categorical_cols]
    num_scaled = [col + '_scaled' for col in numerical_cols]

    return df, cat_ids, num_scaled, encoders, scalers

In [24]:
def make_tf_dataset(df, cat_cols, num_cols, target_col='Listening_Time_minutes', batch_size=32, shuffle=True):
    def gen():
        for _, row in df.iterrows():
            yield (
                {
                    'cat_inputs': [row[col] for col in cat_cols],
                    'num_inputs': [row[col] for col in num_cols],
                },
                row[target_col]
            )

    output_signature = (
        {
            'cat_inputs': tf.TensorSpec(shape=(len(cat_cols),), dtype=tf.int32),
            'num_inputs': tf.TensorSpec(shape=(len(num_cols),), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [25]:
df_preprocessed, cat_cols, num_cols, encs, scalers = preprocess_combined_df(df_combined)
train_df_full = df_preprocessed[df_preprocessed['is_train'] == 1].copy()
test_df = df_preprocessed[df_preprocessed['is_train'] == 0].copy()

In [27]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(train_df_full, test_size=0.2, random_state=42)

In [28]:
train_ds = make_tf_dataset(train_df, cat_cols, num_cols, target_col='Listening_Time_minutes')
val_ds = make_tf_dataset(val_df, cat_cols, num_cols, target_col='Listening_Time_minutes', shuffle=False)
test_ds = make_tf_dataset(test_df, cat_cols, num_cols, target_col='Listening_Time_minutes', shuffle=False)

## Model Training

In [29]:
def build_model(cat_cardinalities, num_numerical):
    cat_inputs = layers.Input(shape=(len(cat_cardinalities),), name='cat_inputs', dtype='int32')
    num_inputs = layers.Input(shape=(num_numerical,), name='num_inputs')

    embeddings = []
    for i, vocab_size in enumerate(cat_cardinalities):
        emb_dim = min(50, (vocab_size + 1) // 2)
        emb = layers.Embedding(input_dim=vocab_size, output_dim=emb_dim)(layers.Lambda(lambda x: x[:, i])(cat_inputs))
        embeddings.append(emb)

    cat_embed = layers.Concatenate()(embeddings)
    x = layers.Concatenate()([cat_embed, num_inputs])
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(1)(x)

    return models.Model(inputs=[cat_inputs, num_inputs], outputs=x)

In [30]:
cat_cardinalities = [df_preprocessed[col].nunique() for col in cat_cols]
model = build_model(cat_cardinalities, num_numerical=len(num_cols))
model.compile(optimizer='adam', loss='mse', metrics=['mae', rmse])

In [31]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[reduce_lr]
)

Epoch 1/20


In [ ]:
def plot_history(history):
  fig, ax2 = plt.subplots(2, figsize=(12, 12))
  ax2.legend(['train', 'validation'], loc='upper left')
  ax2.plot(history.history['loss'])
  ax2.plot(history.history['val_loss'])
  ax2.set_title('model loss')
  ax2.set_ylabel('loss')
  ax2.set_xlabel('epoch')

In [ ]:
plot_history(history)

## Model Evaluation

In [ ]:
# Check for Root Mean Square error and R2 Score
y_pred = model.predict(X_test)
print(f"Root Mean Square Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")